# Pure Weber Planetary Atom: Elliptical Orbit ($\eta = 0.8$, $a = 0$)

This notebook demonstrates a **bound three-body state** in Weber
electrodynamics: two like charges forming a sub-critical nucleus, orbited
by an unlike charge in a wide elliptical orbit.

## Physics Background

### Weber's Velocity-Dependent Force

Weber's force law between two charges $q_i$, $q_j$ separated by distance
$r$ with radial velocity $\dot{r}$ and acceleration $\ddot{r}$ is

$$F = \frac{q_i q_j}{r^2}\left(1 - \frac{\dot{r}^2}{2c^2} + \frac{r\ddot{r}}{c^2}\right)$$

The velocity- and acceleration-dependent terms create an **effective
inertial mass** that depends on separation:

$$\mu_{\text{eff}}(r) = \mu\left(1 - \frac{\rho}{r}\right)$$

where $\mu$ is the reduced mass and $\rho = q_i q_j / (\mu c^2)$ is
the **critical radius**.

### Critical Radius and Sub-Critical Binding

For two like charges ($q_i q_j > 0$), the critical radius $\rho > 0$ is
a real positive distance. The effective inertial mass changes sign at
$r = \rho$, creating two permanently separated dynamical regimes:

- **Distant state** ($r > \rho$): ordinary Coulomb-like repulsion and
  scattering.
- **Molecular state** ($r < \rho$): the particles are bound, oscillating
  between their initial separation $r_0$ and $r = 0$. The sign reversal
  of $\mu_{\text{eff}}$ turns repulsion into effective attraction.

No continuous trajectory can cross $r = \rho$. Weber called this a
"molecular movement" (Sixth Memoir, §9.9).

### The Planetary Atom Model

The **planetary atom** combines a sub-critical nucleus with an orbiting
unlike charge:

| Particle | Charge | Role |
|---|---|---|
| 1 | $+q$ | Nucleus (bound sub-critically with particle 2) |
| 2 | $+q$ | Nucleus (bound sub-critically with particle 1) |
| 3 | $-q$ | Orbiter (Coulomb-attracted to net charge $+2q$) |

The nucleus oscillates at period $T_{\text{nuc}} \approx 2\sqrt{2}\,r_{\text{nuc}}/c$
(fast), while the orbiter has period
$T_{\text{orb}} \approx 2\pi R / v_{\text{circ}}$ (slow). The natural
timescale separation $R \gg r_{\text{nuc}}$ keeps the orbiter far from the
nucleus, preventing three-body instability.

### This Run

At $\eta_{\text{orb}} = 0.8$ (80% of circular speed), the orbiter has
more kinetic energy than the $\eta = 0.7$ case and traces a **wide
elliptical orbit** with apoapsis extending to $\sim 2.5R$. This serves
as the baseline for comparison with the Zöllner notebooks, which use
the same $\eta = 0.8$ but add gravitational enhancement.

**References**: Weber, Sixth Memoir (1871) §§9.8--9.17; Frauenfelder &
Weber, *Anal. Math. Phys.* **14**:31 (2024); see
`docs/theory/CriticalRadiusAndLikeChargeAttraction.md`,
`docs/theory/InitialConditions.md`, and
`docs/exploratory/ThreeBodyBoundStates.md`.

In [ ]:
using WeberElectrodynamics
using LinearAlgebra
using Plots
using Printf

## 1. System Construction and Physical Parameters

In [ ]:
# Physical parameters
m = 1.0               # equal masses
q_pos = 1.0           # positive charges (nucleus)
q_neg = -1.0          # negative charge (orbiter)
c = 4.0

# Derived quantities
mu_nuc = m * m / (m + m)          # nucleus reduced mass = 0.5
rho = q_pos^2 / (mu_nuc * c^2)   # critical radius = 0.125
M_nuc = 2m                         # nucleus total mass
mu_orb = M_nuc * m / (M_nuc + m)  # orbiter reduced mass = 2/3

# Nucleus parameters
r_nuc = 0.05   # sub-critical nucleus separation
T_nuc = 2 * sqrt(2) * r_nuc / c   # nucleus oscillation period

# Orbiter parameters
R = 1.0        # orbiter distance from nucleus COM
Q_eff = 2.0    # effective charge seen by orbiter (q1 + q2)
v_circ = sqrt(Q_eff / (mu_orb * R))  # circular orbit speed
T_orb = 2 * pi * R / v_circ          # orbiter orbital period

# Integration parameters
dt = 1e-4
bounce_r = 0.02

system = WeberSystem(3, 2)

@printf("Three-body planetary atom:\n")
@printf("  Particles:  %d (2D)\n", system.n_particles)
@printf("  DOF:        %d\n", system.degrees_of_freedom)
@printf("\nNucleus (particles 1, 2):\n")
@printf("  q1 = q2 = +%.1f, m = %.1f\n", q_pos, m)
@printf("  r_nuc = %.4f < rho = %.4f\n", r_nuc, rho)
@printf("  T_nuc ~ %.6f\n", T_nuc)
@printf("\nOrbiter (particle 3):\n")
@printf("  q3 = %.1f, m3 = %.1f\n", q_neg, m)
@printf("  R = %.2f, mu_orb = %.4f\n", R, mu_orb)
@printf("  v_circ = %.4f, T_orb ~ %.4f\n", v_circ, T_orb)
@printf("  T_orb / T_nuc = %.1f  (timescale separation)\n", T_orb / T_nuc)
@printf("\nIntegration:\n")
@printf("  dt = %.0e, bounce_r = %.2f, c = %.1f\n", dt, bounce_r, c)

## 2. Initial Condition Construction

The planetary atom IC follows the recipe from `docs/theory/InitialConditions.md` §9:

1. **Nucleus** (particles 1, 2): placed at $\pm r_{\text{nuc}}/2$ along the
   $x$-axis with zero momenta (at the outer turning point of the sub-critical
   oscillation).
2. **Orbiter** (particle 3): placed at $(0, R)$ with tangential momentum
   $p_{3x} = m_3 \cdot \eta \cdot v_{\text{circ}}$.
3. **COM and momentum conservation**: all positions shifted to place COM at
   origin; nucleus momenta adjusted so $\sum_i \vec{p}_i = 0$.

In [ ]:
function make_planetary_atom_ic(r_nuc, R, eta_orb, m, q_pos, q_neg, c)
    # Positions (before COM adjustment)
    x1 = -r_nuc / 2;  y1 = 0.0
    x2 = +r_nuc / 2;  y2 = 0.0
    x3 = 0.0;          y3 = R

    # Orbiter circular speed estimate
    M_nuc = 2m
    mu_orb = M_nuc * m / (M_nuc + m)
    Q_eff = 2 * abs(q_pos * q_neg)  # |q1*q3| + |q2*q3|
    v_circ = sqrt(Q_eff / (mu_orb * R))
    v_orb = eta_orb * v_circ

    # Momenta: orbiter tangential (x-direction at (0, R))
    px3 = m * v_orb
    py3 = 0.0

    # Zero total momentum: distribute recoil to nucleus
    px1 = -px3 / 2;  py1 = 0.0
    px2 = -px3 / 2;  py2 = 0.0

    # COM correction
    M_total = 3m
    cx = (m * x1 + m * x2 + m * x3) / M_total
    cy = (m * y1 + m * y2 + m * y3) / M_total

    q0 = [x1 - cx, y1 - cy, x2 - cx, y2 - cy, x3 - cx, y3 - cy]
    p0 = [px1, py1, px2, py2, px3, py3]

    return q0, p0, v_circ, v_orb
end

# Helper: compute all pair separations from solution
function compute_pair_separations(sol)
    nt = length(sol.t)
    r12 = zeros(nt); r13 = zeros(nt); r23 = zeros(nt)
    for k in 1:nt
        qk = sol.q[k]
        r12[k] = sqrt((qk[1] - qk[3])^2 + (qk[2] - qk[4])^2)
        r13[k] = sqrt((qk[1] - qk[5])^2 + (qk[2] - qk[6])^2)
        r23[k] = sqrt((qk[3] - qk[5])^2 + (qk[4] - qk[6])^2)
    end
    return r12, r13, r23
end

# Helper: plot all pair separations
function plot_pair_separations(t, r12, r13, r23, rho, title_str)
    plt = plot(;
        title = title_str,
        xlabel = "Time t",
        ylabel = "Pair separation",
        legend = :outertopright,
        framestyle = :box,
        grid = true, gridalpha = 0.2,
        size = (1200, 500),
    )
    plot!(plt, t, r12, label = "r\u2081\u2082 (nucleus)", linewidth = 1.5, color = :steelblue)
    plot!(plt, t, r13, label = "r\u2081\u2083 (orb-nuc1)", linewidth = 1, color = :firebrick, alpha = 0.7)
    plot!(plt, t, r23, label = "r\u2082\u2083 (orb-nuc2)", linewidth = 1, color = :forestgreen, alpha = 0.7)
    hline!(plt, [rho], linestyle = :dash, linewidth = 2, color = :black,
        label = @sprintf("\u03c1 = %.3f", rho))
    return plt
end

println("Helpers defined.")

## 3. Solve

In [ ]:
eta = 0.8
tmax = 100.0

q0, p0, vc, vorb = make_planetary_atom_ic(r_nuc, R, eta, m, q_pos, q_neg, c)

prob = WeberProblem(system, (0.0, tmax), q0, p0;
    masses = [m, m, m], charges = [q_pos, q_pos, q_neg], c = c, dt = dt,
    regularization_enabled = false,
    regularization_collision_bounce_radius = bounce_r,
    zollner_enabled = false, zollner_a = 0.0)

sol = solve(prob)

@printf("Run 2: Pure Weber, eta = %.1f, a = 0\n", eta)
@printf("  retcode: %s, steps: %d\n", sol.retcode, length(sol.t))
@printf("  v_orb = %.4f (v_circ = %.4f)\n", vorb, vc)

## 4. Diagnostics

In [ ]:
traj = compute_trajectory_data(sol, 3, 2; stride = 1)
energy = compute_energy_timeseries(sol; stride = 1)
momentum = compute_momentum_timeseries(sol; stride = 1)
r12, r13, r23 = compute_pair_separations(sol)

@printf("Diagnostics:\n")
@printf("  E(0) = %.6f\n", energy.total_energy[1])
@printf("  Energy error (%%): %.4f\n", energy.statistics.global_error_percent_max)
@printf("  Nucleus r\u2081\u2082: [%.4e, %.4f]  (rho = %.4f) %s\n",
    minimum(r12), maximum(r12), rho,
    maximum(r12) < rho ? "INTACT" : "BROKEN")
@printf("  Orbiter r\u2081\u2083: [%.4f, %.4f]\n", minimum(r13), maximum(r13))
@printf("  Orbiter r\u2082\u2083: [%.4f, %.4f]\n", minimum(r23), maximum(r23))

## 5. Plots

In [ ]:
plot_trajectories(traj)

In [ ]:
plot_pair_separations(sol.t, r12, r13, r23, rho,
    "Pair Separations (\u03b7=0.8, a=0)")

In [ ]:
plot_energy(energy)

In [ ]:
plot_energy_errors(energy)

In [ ]:
plot_momentum(momentum)

In [ ]:
# Phase space for the nucleus pair (1,2)
forces_12 = compute_pair_force_timeseries(sol, (1, 2), 3, 2,
    [m, m, m], [q_pos, q_pos, q_neg], c; stride = 1)
plot_phase_space(forces_12)

In [ ]:
# Phase space for an orbiter pair (1,3)
forces_13 = compute_pair_force_timeseries(sol, (1, 3), 3, 2,
    [m, m, m], [q_pos, q_pos, q_neg], c; stride = 1)
plot_phase_space(forces_13)

## 6. Conclusions

This simulation demonstrates the **pure Weber planetary atom** at
$\eta = 0.8$ (80% of circular speed, no Zöllner extension).

### Key Observations

1. **Nucleus integrity**: The nucleus pair separation $r_{12}$ stays well
   below the critical radius $\rho = 0.125$, confirming the sub-critical
   binding is robust even over 100 time units ($\sim 28$ orbiter periods).

2. **Wide elliptical orbit**: Unlike the near-circular $\eta = 0.7$ case,
   the orbiter at $\eta = 0.8$ traces a significantly elliptical orbit
   with apoapsis $\sim 2.5R$ -- the orbiter's extra kinetic energy
   carries it much farther from the nucleus before falling back.

3. **Baseline for Zöllner comparison**: This wide ellipse at $\eta = 0.8$
   is the reference case. The Zöllner extension ($a > 0$) strengthens
   unlike-pair coupling, progressively tightening and circularizing this
   orbit:

   | $a$ | Orbiter range | Shape |
   |---|---|---|
   | 0.0 | $[1.0, 2.5]$ | Wide ellipse (this notebook) |
   | 0.1 | $[1.0, 1.8]$ | Moderate ellipse |
   | 0.5 | $[0.85, 1.03]$ | Near-circular |

4. **Phase space structure**: The orbiter phase portrait reveals the
   elliptical character -- a wider loop in $(r, \dot{r})$ space compared
   to the tight ellipse at $\eta = 0.7$.

### Context in the Series

- `weber_near_circular_orbit.ipynb`: Pure Weber, near-circular ($\eta = 0.7$, $a = 0$)
- `weber_elliptical_orbit.ipynb`: Pure Weber, wide ellipse ($\eta = 0.8$, $a = 0$)
- `zollner_moderate_circularization.ipynb`: Zöllner $a = 0.1$ tightens the orbit
- `zollner_strong_circularization.ipynb`: Zöllner $a = 0.5$ circularizes the orbit

See `docs/exploratory/ThreeBodyBoundStates.md` for the full parameter study.

## 7. Rolling Animation Viewer

Interactive Makie animation replaying the pre-computed solution with
rolling trajectories, energy, momentum, angular momentum, and phase space.

Load any Makie backend (GLMakie for native window, WGLMakie for browser,
CairoMakie for static output) before calling `animate_weber`.

In [ ]:
using GLMakie  # or CairoMakie, WGLMakie
animate_weber(sol; buffer_size = 2000, tail_length = 200, compute_batch = 50)